# RoBERTa Embeddings + Traditional ML Classification

Experiment: Check text length, encode directly with RoBERTa, train ML classifiers.
Pipeline:
1. Analyze text lengths in HateXplain
2. RoBERTa encodes text → dense embeddings
3. Train ML classifiers (XGBoost, MLP, etc.) on embeddings

In [1]:
import json
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
# Hugging Face transformers for loading pre-trained RoBERTa tokenizer and model
from transformers import RobertaTokenizer, RobertaModel
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
# ML classifiers that will operate on the extracted embeddings
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Use GPU if available for faster RoBERTa inference, otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# RoBERTa-base outputs 768-dimensional embeddings for each token
EMBEDDING_DIM = 768

print(f"Device: {DEVICE}")

Device: cpu


In [3]:
def load_hatexplain_data(file_path):
    """Read JSONL file where each line is a JSON object representing one data sample."""
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def reconstruct_text(post_tokens):
    """HateXplain stores text as a list of tokens; join them back into a single string."""
    return ' '.join(post_tokens)

# Load the three splits from the HateXplain dataset
train_data = load_hatexplain_data('hatexplain/hatexplain_train.jsonl')
valid_data = load_hatexplain_data('hatexplain/hatexplain_validation.jsonl')
test_data = load_hatexplain_data('hatexplain/hatexplain_test.jsonl')

print(f"Train: {len(train_data)}, Valid: {len(valid_data)}, Test: {len(test_data)}")

Train: 15383, Valid: 1922, Test: 1924


In [4]:
# Analyze text lengths to understand the data distribution
# This helps verify that texts fit within RoBERTa's 512-token context window
all_data = train_data + valid_data + test_data

# Count character-level lengths for each post
text_lengths = []
for item in all_data:
    text = reconstruct_text(item['post_tokens'])
    text_lengths.append(len(text))

# Count word-level lengths (number of tokens in the original tokenization)
word_counts = [len(item['post_tokens']) for item in all_data]

print("=== Text Length Analysis ===")
print(f"Character lengths - Mean: {np.mean(text_lengths):.1f}, Median: {np.median(text_lengths):.1f}")
print(f"Character lengths - Min: {min(text_lengths)}, Max: {max(text_lengths)}")
print(f"\nWord counts - Mean: {np.mean(word_counts):.1f}, Median: {np.median(word_counts):.1f}")
print(f"Word counts - Min: {min(word_counts)}, Max: {max(word_counts)}")

# Bucket distribution shows how many posts fall into each word-count range
print("\n=== Word Count Distribution ===")
print(f"<= 10 words: {sum(1 for w in word_counts if w <= 10)}")
print(f"11-30 words: {sum(1 for w in word_counts if 11 <= w <= 30)}")
print(f"31-50 words: {sum(1 for w in word_counts if 31 <= w <= 50)}")
print(f"51-100 words: {sum(1 for w in word_counts if 51 <= w <= 100)}")
print(f"> 100 words: {sum(1 for w in word_counts if w > 100)}")

=== Text Length Analysis ===
Character lengths - Mean: 126.4, Median: 108.0
Character lengths - Min: 8, Max: 526

Word counts - Mean: 23.4, Median: 20.0
Word counts - Min: 2, Max: 165

=== Word Count Distribution ===
<= 10 words: 3916
11-30 words: 9591
31-50 words: 5282
51-100 words: 439
> 100 words: 1
Character lengths - Mean: 126.4, Median: 108.0
Character lengths - Min: 8, Max: 526

Word counts - Mean: 23.4, Median: 20.0
Word counts - Min: 2, Max: 165

=== Word Count Distribution ===
<= 10 words: 3916
11-30 words: 9591
31-50 words: 5282
51-100 words: 439
> 100 words: 1


In [5]:
# Load the pre-trained RoBERTa tokenizer and model from Hugging Face
# The tokenizer converts raw text into subword token IDs that the model expects
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
# The base model (without a classification head) outputs contextual embeddings
roberta_model = RobertaModel.from_pretrained('roberta-base').to(DEVICE)
# Set model to evaluation mode — disables dropout for deterministic embeddings
roberta_model.eval()
print("RoBERTa model loaded")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1919.51it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model loaded


In [ ]:
def get_roberta_embedding(text, tokenizer, model, device,
                          pooling='cls', rationale_scores=None):
    """Extract a fixed-size 768-dim embedding from RoBERTa.

    pooling strategies
    ------------------
    'cls'
        Use the <s> (CLS) token at position 0.  After all transformer layers
        this token has attended to every other token, so its hidden state is a
        summary of the whole sequence.  Simple and fast.

    'mean'
        Average the hidden states of every REAL (non-padding) token.
        Tends to work slightly better than CLS for semantic similarity tasks
        because it integrates signal from all tokens equally.

    'rationale'
        Weighted mean pooling using human annotator rationale scores as weights.
        Tokens the annotators flagged as evidence (score > 0) contribute more
        to the embedding; tokens they ignored contribute less.

        Why this is useful for hate speech:
          "i love everyone except those dirty [slur]"
          CLS/mean spread weight across all 8 tokens equally.
          Rationale weighting concentrates weight on "[slur]" (score=1.0)
          which is actually what makes the post hateful.
          The resulting embedding is more discriminative for the classifier.

        Alignment challenge:
          HateXplain rationales are word-level (one score per whitespace token).
          RoBERTa tokenizes into subwords, e.g. "unhappy" → ["un", "happy"].
          We align by assigning each subword the score of its parent word.

        Falls back to mean pooling if no rationale scores are provided or
        they are all zero (e.g. normal-labelled samples have empty rationales).

    Parameters
    ----------
    text             : raw post string
    tokenizer        : RoBERTa tokenizer
    model            : RoBERTa model in eval mode
    device           : torch device
    pooling          : 'cls' | 'mean' | 'rationale'
    rationale_scores : list of floats, one per whitespace token in `text`.
                       Only used when pooling='rationale'.
    """
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512,        # includes the 2 special tokens (<s> and </s>)
        padding=True,
        return_offsets_mapping=False,
    )
    inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs_on_device)

    # hidden: (1, seq_len, 768)
    hidden = outputs.last_hidden_state  # stays on device
    attention_mask = inputs['attention_mask']  # 1 for real tokens, 0 for padding

    if pooling == 'cls':
        # <s> token is always at index 0
        embedding = hidden[0, 0, :].cpu().numpy()

    elif pooling == 'mean':
        # Average over real tokens only (mask out padding)
        mask = attention_mask[0].unsqueeze(-1).float().to(device)  # (seq_len, 1)
        summed = (hidden[0] * mask).sum(dim=0)                     # (768,)
        embedding = (summed / mask.sum()).cpu().numpy()

    elif pooling == 'rationale':
        # --- Build subword-level weight vector from word-level rationale scores ---
        # Tokenize without special tokens to get the subword → word mapping
        words = text.split()
        word_ids = []
        for word_idx, word in enumerate(words):
            subword_ids = tokenizer.encode(word, add_special_tokens=False)
            word_ids.extend([word_idx] * len(subword_ids))

        # The full token sequence is: <s>, *subwords, </s>
        # so shift word_ids by 1 (for <s>) and cap at max_length - 2
        max_subword_tokens = inputs['input_ids'].shape[1] - 2  # excl. <s> and </s>
        word_ids_trimmed = word_ids[:max_subword_tokens]

        # Build weight vector aligned to the full token sequence
        # Use rationale_scores if available and non-zero, else fall back to uniform
        scores = rationale_scores if (rationale_scores and any(s > 0 for s in rationale_scores)) else None

        weights = []
        for w_idx in word_ids_trimmed:
            if scores is not None and w_idx < len(scores):
                weights.append(scores[w_idx])
            else:
                weights.append(1.0)  # uniform fallback

        # Pad weight vector to match full sequence: <s>=0, subwords, </s>=0, [PAD]=0
        seq_len = inputs['input_ids'].shape[1]
        full_weights = [0.0] + weights + [0.0] * (seq_len - len(weights) - 1)
        full_weights = full_weights[:seq_len]

        weight_tensor = torch.tensor(full_weights, dtype=torch.float32).to(device)

        # Zero out padding positions using attention mask
        att = attention_mask[0].float().to(device)
        weight_tensor = weight_tensor * att

        weight_sum = weight_tensor.sum()
        if weight_sum == 0:
            # All weights zero (e.g. normal sample with no rationales) → mean pool
            mask = att.unsqueeze(-1)
            summed = (hidden[0] * mask).sum(dim=0)
            embedding = (summed / mask.sum()).cpu().numpy()
        else:
            w = weight_tensor.unsqueeze(-1)          # (seq_len, 1)
            embedding = ((hidden[0] * w).sum(dim=0) / weight_sum).cpu().numpy()

    else:
        raise ValueError(f"Unknown pooling mode '{pooling}'. Use 'cls', 'mean', or 'rationale'.")

    return embedding.flatten()  # (768,)

In [ ]:
def encode_dataset(data, tokenizer, model, device,
                   pooling='rationale', sample_size=None):
    """Encode every sample into a RoBERTa embedding and extract all HateXplain metadata.

    pooling
    -------
    'cls'       — Use the <s> (CLS) token hidden state.
    'mean'      — Mean-pool all real token hidden states.
    'rationale' — Weighted mean weighted by human annotator rationale scores.
                  Falls back to mean pooling for samples with all-zero rationales
                  (i.e. normal-labelled posts where annotators gave no evidence).

    Fields extracted per sample
    ---------------------------
    id               : original sample ID
    source           : platform — 'gab' or 'twitter' (parsed from ID suffix)
    text             : reconstructed post text
    word_count       : number of whitespace tokens in the original post
    embedding        : 768-dim embedding (strategy set by `pooling`)
    label            : majority-vote class  (0=hate, 1=offensive, 2=normal)
    agreement        : fraction of annotators who voted for the majority label
                       e.g. 1.0 = unanimous, 0.33 = one out of three agreed
                       Low agreement = ambiguous sample; worth tracking separately
    majority_target  : most-cited community targeted by the post
                       e.g. 'African', 'Islam', 'Women', 'None'
                       Aggregated across all annotators' target lists
    rationale_scores : per-token importance score averaged across all annotators
                       Shape: (word_count,), values in [0.0, 1.0]
                       0.0 = no annotator highlighted this token
                       1.0 = every annotator flagged this token as evidence
    highlighted_tokens: tokens with rationale_score > 0 — the words annotators
                        felt triggered or evidenced the classification
    """
    if sample_size:
        data = data[:sample_size]

    results = []

    for item in tqdm(data, desc=f"Encoding ({pooling})"):
        tokens = item['post_tokens']
        text = reconstruct_text(tokens)
        word_count = len(tokens)

        # --- Label: majority vote across annotators ---
        labels = item['annotators']['label']
        majority_label = max(set(labels), key=labels.count)

        # Agreement = fraction who voted for majority. Range: [0.33, 1.0] with 3 annotators.
        agreement = labels.count(majority_label) / len(labels)

        # --- Target community: most commonly cited across all annotator target lists ---
        all_targets = [t for t_list in item['annotators']['target'] for t in t_list]
        specific_targets = [t for t in all_targets if t != 'None']
        if specific_targets:
            majority_target = max(set(specific_targets), key=specific_targets.count)
        else:
            majority_target = 'None'

        # --- Source platform: encoded in the ID suffix ---
        source = item['id'].split('_')[-1]
        if source not in ('gab', 'twitter'):
            source = 'unknown'

        # --- Rationale scores: per-token evidence strength ---
        # Filter to only annotator rationale arrays that match the token count.
        # Some HateXplain samples have annotators with mismatched rationale lengths;
        # np.mean requires a rectangular array, so drop any that don't align.
        rationales = item['rationales']
        valid_rationales = [r for r in rationales if len(r) == word_count]
        if valid_rationales:
            rationale_scores = np.mean(valid_rationales, axis=0).tolist()
        else:
            rationale_scores = [0.0] * word_count

        highlighted_tokens = [t for t, s in zip(tokens, rationale_scores) if s > 0]

        # --- Embedding: pooling strategy selected by caller ---
        embedding = get_roberta_embedding(
            text, tokenizer, model, device,
            pooling=pooling,
            rationale_scores=rationale_scores,
        )

        results.append({
            'id':                item['id'],
            'source':            source,
            'text':              text,
            'word_count':        word_count,
            'embedding':         embedding,
            'label':             majority_label,
            'agreement':         agreement,
            'majority_target':   majority_target,
            'rationale_scores':  rationale_scores,
            'highlighted_tokens': highlighted_tokens,
        })

    return results


In [ ]:
# Encode all three splits with BOTH pooling strategies so we can compare them.
# Set SAMPLE_SIZE to a small integer (e.g. 500) for quick experimentation.
SAMPLE_SIZE = None

POOLING_MODES = ['cls', 'rationale']   # 'mean' can be added for a third comparison

results_by_pooling = {}
for mode in POOLING_MODES:
    print(f"\n=== Pooling: {mode} ===")
    results_by_pooling[mode] = {
        split: encode_dataset(data, roberta_tokenizer, roberta_model, DEVICE,
                              pooling=mode, sample_size=SAMPLE_SIZE)
        for split, data in [('train', train_data),
                             ('valid', valid_data),
                             ('test',  test_data)]
    }

# Convenience aliases pointing at the rationale-weighted results (primary strategy)
train_results = results_by_pooling['rationale']['train']
valid_results = results_by_pooling['rationale']['valid']
test_results  = results_by_pooling['rationale']['test']

print(f"\nReady: {len(train_results)} train / {len(valid_results)} valid / {len(test_results)} test")

Encoding training data...


Encoding: 100%|██████████| 15383/15383 [09:49<00:00, 26.11it/s]


Encoding validation data...


Encoding: 100%|██████████| 1922/1922 [00:55<00:00, 34.57it/s]


Encoding test data...


Encoding: 100%|██████████| 1924/1924 [00:44<00:00, 42.98it/s]


In [ ]:
# Inspect the enriched metadata to understand what we now have per sample
all_results = train_results + valid_results + test_results

print("=== Sample enriched record ===")
sample = next(r for r in train_results if r['agreement'] < 1.0 and r['highlighted_tokens'])
print(f"Text:               {sample['text'][:80]}...")
print(f"Label:              {sample['label']}  (0=hate, 1=offensive, 2=normal)")
print(f"Agreement:          {sample['agreement']:.2f}  ({sample['agreement']*3:.0f}/3 annotators agreed)")
print(f"Source:             {sample['source']}")
print(f"Majority target:    {sample['majority_target']}")
print(f"Highlighted tokens: {sample['highlighted_tokens']}")
print()

# --- Agreement distribution ---
print("=== Annotator agreement distribution (train) ===")
agreements = [r['agreement'] for r in train_results]
for threshold, label in [(1.0, "unanimous (3/3)"), (2/3, "majority (2/3)"), (1/3, "minority (1/3)")]:
    count = sum(1 for a in agreements if round(a, 2) == round(threshold, 2))
    print(f"  {label}: {count} ({100*count/len(agreements):.1f}%)")
print()

# --- Source distribution ---
print("=== Source platform distribution ===")
sources = pd.Series([r['source'] for r in all_results])
print(sources.value_counts().to_string())
print()

# --- Target community distribution (hate samples only) ---
print("=== Top target communities (hate-labelled samples only) ===")
hate_targets = [r['majority_target'] for r in all_results if r['label'] == 0]
target_counts = pd.Series(hate_targets).value_counts()
print(target_counts.head(10).to_string())
print()

# --- Rationale coverage ---
has_rationale = sum(1 for r in all_results if any(s > 0 for s in r['rationale_scores']))
print(f"=== Rationale coverage ===")
print(f"Samples with at least one highlighted token: {has_rationale}/{len(all_results)} "
      f"({100*has_rationale/len(all_results):.1f}%)")

In [ ]:
def prepare_data(results):
    """Stack embeddings into (n_samples, 768) and labels into (n_samples,)."""
    X = np.array([r['embedding'] for r in results])
    y = np.array([r['label'] for r in results])
    word_counts = [r['word_count'] for r in results]
    return X, y, word_counts

# Primary arrays — rationale-weighted pooling
X_train, y_train, train_word_counts = prepare_data(train_results)
X_valid, y_valid, valid_word_counts = prepare_data(valid_results)
X_test,  y_test,  test_word_counts  = prepare_data(test_results)

# CLS arrays — used in the pooling comparison cell below
X_train_cls, _, _ = prepare_data(results_by_pooling['cls']['train'])
X_valid_cls, _, _ = prepare_data(results_by_pooling['cls']['valid'])
X_test_cls,  _, _ = prepare_data(results_by_pooling['cls']['test'])

print(f"X_train shape (rationale): {X_train.shape}")
print(f"X_train shape (cls):       {X_train_cls.shape}")
print(f"Label distribution (train): {pd.Series(y_train).value_counts().to_dict()}")

X_train shape: (15383, 768)
X_test shape: (1924, 768)
Label distribution (train): {1: 6251, 0: 4748, 2: 4384}


In [ ]:
# ── Pooling strategy comparison ─────────────────────────────────────────────
# Run the same classifiers on CLS vs rationale-weighted embeddings side by side.
# Rationale-weighted pooling focuses the embedding on the tokens annotators
# flagged as evidence, so it should carry a stronger discriminative signal.

from copy import deepcopy
from sklearn.linear_model import LogisticRegression as LR_cmp
from xgboost import XGBClassifier as XGB_cmp

comparison_classifiers = {
    'XGBoost':             XGB_cmp(
                               n_estimators=100, learning_rate=0.1,
                               eval_metric='mlogloss', random_state=42),
    'Logistic Regression': LR_cmp(max_iter=1000, random_state=42),
}

pooling_data = {
    'cls':       (X_train_cls, X_test_cls),
    'rationale': (X_train,     X_test),
}

rows = []
for clf_name, clf in comparison_classifiers.items():
    for pool_name, (Xtr, Xte) in pooling_data.items():
        m = deepcopy(clf)
        m.fit(Xtr, y_train)
        acc = m.score(Xte, y_test)
        rows.append({'Classifier': clf_name, 'Pooling': pool_name, 'Test Accuracy': acc})
        print(f"  {clf_name:22s} [{pool_name:9s}]  acc={acc:.4f}")

comparison_df = pd.DataFrame(rows)
pivot = comparison_df.pivot(index='Classifier', columns='Pooling', values='Test Accuracy')
pivot['delta (rationale − cls)'] = pivot['rationale'] - pivot['cls']
pivot = pivot[['cls', 'rationale', 'delta (rationale − cls)']].round(4)

print("\n=== Pooling strategy comparison ===")
print(pivot.to_string())
best_pool = 'rationale' if pivot['rationale'].mean() >= pivot['cls'].mean() else 'cls'
print(f"\nBetter average pooling: {best_pool}")


In [10]:
# Generic helper to train a classifier on embeddings and evaluate on the test set
def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name):
    """Fit the model on training embeddings, predict on test, and print metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"\n=== {model_name} ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    # Per-class precision, recall, F1 for all three classes
    print(classification_report(y_test, y_pred, target_names=['hate', 'offensive', 'normal']))
    
    return accuracy_score(y_test, y_pred), y_pred

In [ ]:
# 1. XGBoost — gradient-boosted decision trees
# Ref: https://xgboost.readthedocs.io/en/stable/parameter.html
xgb_model = xgb.XGBClassifier(
    # --- General Parameters ---
    # booster='gbtree',     # Which booster to use. Options:
    #                       #   'gbtree'   — tree-based models (default, best for most tasks)
    #                       #   'dart'     — tree-based with dropout (reduces overfitting)
    #                       #   'gblinear' — linear model (like regularized logistic regression)
    
    # --- Tree Booster Parameters ---
    n_estimators=100,       # Number of boosting rounds (trees). Try: 50, 200, 500, 1000
                            #   More trees = better fit but slower & risk overfitting
    max_depth=6,            # Max tree depth [default=6]. Try: 3, 4, 8, 10. 0 = no limit
                            #   Shallow trees generalize better; deep trees capture complex patterns
                            #   Deeper trees aggressively consume memory
    learning_rate=0.1,      # Step size shrinkage [default=0.3, alias: eta]. Range: [0, 1]
                            #   Try: 0.01, 0.05, 0.1, 0.3
                            #   Lower = needs more trees but often better generalization
    random_state=42,
    eval_metric='mlogloss', # Evaluation metric for multiclass. Options:
                            #   'mlogloss' — multiclass log-loss (default for multiclass)
                            #   'merror'   — multiclass misclassification rate
                            #   'auc'      — ROC-AUC (1-vs-rest, needs multi:softprob objective)
                            #   'aucpr'    — precision-recall AUC

    # --- Regularization & Sampling ---
    # min_child_weight=1,   # Min sum of instance weight (hessian) in a child [default=1]
    #                       #   Try: 1, 3, 5, 10. Range: [0, inf]
    #                       #   Higher = more conservative, prevents learning noise
    # gamma=0,              # Min loss reduction for a split [default=0, alias: min_split_loss]
    #                       #   Try: 0, 0.1, 0.5, 1.0. Range: [0, inf]
    #                       #   Acts as pruning — higher gamma = fewer splits
    # subsample=1.0,        # Fraction of training samples per tree [default=1]. Range: (0, 1]
    #                       #   Try: 0.6, 0.8. Subsampling reduces overfitting
    # sampling_method='uniform',  # How to sample training instances. Options:
    #                             #   'uniform'        — equal probability (default)
    #                             #   'gradient_based' — proportional to gradient magnitude
    #                             #                      allows subsample as low as 0.1
    # colsample_bytree=1.0, # Fraction of features per tree [default=1]. Range: (0, 1]
    #                       #   Try: 0.6, 0.8. Lower = more regularization
    # colsample_bylevel=1.0,# Fraction of features per tree depth level [default=1]
    # colsample_bynode=1.0, # Fraction of features per split [default=1]
    #                       #   These three stack multiplicatively:
    #                       #   bytree=0.5 * bylevel=0.5 * bynode=0.5 with 768 features
    #                       #   = 96 features at each split
    # reg_alpha=0,          # L1 regularization on weights [default=0, alias: alpha]
    #                       #   Try: 0, 0.01, 0.1, 1. Encourages sparsity
    # reg_lambda=1,         # L2 regularization on weights [default=1, alias: lambda]
    #                       #   Try: 0.1, 1, 5, 10. Penalizes large weights
    # max_delta_step=0,     # Max delta step per leaf output [default=0, 0 = no constraint]
    #                       #   Try: 1-10 for imbalanced classification
    # scale_pos_weight=1,   # Balance positive/negative weights [default=1]
    #                       #   Less useful for multiclass — use sample_weight instead
    # --- Tree Construction ---
    # tree_method='auto',   # Tree construction algorithm [default='auto']. Options:
    #                       #   'auto'   — same as 'hist'
    #                       #   'exact'  — exact greedy, enumerates all splits (slow but optimal)
    #                       #   'approx' — approximate greedy using quantile sketch
    #                       #   'hist'   — faster histogram-based approximate (recommended)
    # grow_policy='depthwise',  # How new nodes are added [default='depthwise']
    #                           #   'depthwise' — split nodes closest to root (standard)
    #                           #   'lossguide' — split node with highest loss change (like LightGBM)
    #                           #   Only works with hist/approx tree_method
    max_leaves=0,         # Max leaf nodes [default=0, 0 = no limit]
    #                       #   Use with grow_policy='lossguide' for leaf-wise growth
    # max_bin=256,          # Max bins for histogram [default=256]. Only for hist/approx
    #                       #   More bins = better splits but slower
    # --- DART booster params (if booster='dart') ---
    # rate_drop=0.0,        # Dropout rate [default=0.0]. Range: [0.0, 1.0]
    # sample_type='uniform',# How dropped trees are selected: 'uniform' or 'weighted'
    # normalize_type='tree',# Normalization: 'tree' or 'forest'
)
xgb_acc, xgb_pred = train_and_evaluate(xgb_model, X_train, y_train, X_test, y_test, "XGBoost")


=== XGBoost ===
Accuracy: 0.6034
              precision    recall  f1-score   support

        hate       0.62      0.68      0.65       594
   offensive       0.64      0.74      0.68       782
      normal       0.50      0.32      0.39       548

    accuracy                           0.60      1924
   macro avg       0.58      0.58      0.57      1924
weighted avg       0.59      0.60      0.59      1924



In [12]:
# 2. MLP (Multi-Layer Perceptron) — a simple feedforward neural network
# Three hidden layers (256 → 128 → 64) with ReLU activations
# early_stopping monitors validation loss and stops when it plateaus
mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    max_iter=500,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1  # hold out 10% of training data for early stopping
)
mlp_acc, mlp_pred = train_and_evaluate(mlp_model, X_train, y_train, X_test, y_test, "MLP Neural Network")


=== MLP Neural Network ===
Accuracy: 0.6154
              precision    recall  f1-score   support

        hate       0.69      0.59      0.64       594
   offensive       0.64      0.78      0.70       782
      normal       0.48      0.41      0.44       548

    accuracy                           0.62      1924
   macro avg       0.60      0.59      0.59      1924
weighted avg       0.61      0.62      0.61      1924



In [ ]:
# 2b. PyTorch MLP — same simple architecture, but with more activation choices
# sklearn's MLPClassifier only supports: relu, tanh, logistic, identity
# PyTorch gives access to modern activations like GELU, SiLU, LeakyReLU, Mish, etc.
# More useful here in multiclass — harder task benefits more from better activations

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

class SimpleMLP(nn.Module):
    def __init__(self, input_dim=768, hidden_dims=(256, 128, 64), num_classes=3,
                 activation='relu'):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        # Map string name to PyTorch activation module
        activations = {
            'relu':      nn.ReLU(),       # Fast, works well. Can have "dying neurons"
            'tanh':      nn.Tanh(),       # Outputs [-1,1]. Good for centered data
            'sigmoid':   nn.Sigmoid(),    # Outputs [0,1]. Vanishing gradients in deep nets
            'leaky_relu':nn.LeakyReLU(),  # Like ReLU but allows small negatives — fixes dying neurons
            'gelu':      nn.GELU(),       # Used in BERT/RoBERTa/GPT. Smooth, state-of-art for NLP
            'silu':      nn.SiLU(),       # x*sigmoid(x), aka Swish. Smooth, often beats ReLU
            'mish':      nn.Mish(),       # x*tanh(softplus(x)). Strong in vision & NLP
            'elu':       nn.ELU(),        # Smooth for negatives. Faster convergence than ReLU
            'softplus':  nn.Softplus(),   # Smooth approximation of ReLU. No dead neurons
        }
        act_fn = activations[activation]
        
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(act_fn)
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

def train_pytorch_mlp(X_train, y_train, X_test, y_test, activation='gelu',
                       num_classes=3, target_names=['hate', 'offensive', 'normal'],
                       epochs=50, lr=1e-3, batch_size=64):
    """Train a PyTorch MLP and evaluate on test set."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    model = SimpleMLP(num_classes=num_classes, activation=activation).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(epochs):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X_test, dtype=torch.float32).to(device)
        preds = model(X_t).argmax(dim=1).cpu().numpy()
    
    acc = accuracy_score(y_test, preds)
    print(f"\n=== PyTorch MLP (activation={activation}) ===")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=target_names))
    return acc

# Try GELU — the activation used inside RoBERTa/BERT itself
# Change 'gelu' to any of: relu, tanh, sigmoid, leaky_relu, silu, mish, elu, softplus
pytorch_mlp_acc = train_pytorch_mlp(X_train, y_train, X_test, y_test, activation='gelu')

In [13]:
# 3. Logistic Regression — linear classifier, serves as a strong baseline
# max_iter=1000 ensures convergence on high-dimensional (768-d) input
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_acc, lr_pred = train_and_evaluate(lr_model, X_train, y_train, X_test, y_test, "Logistic Regression")


=== Logistic Regression ===
Accuracy: 0.6299
              precision    recall  f1-score   support

        hate       0.64      0.70      0.67       594
   offensive       0.65      0.77      0.71       782
      normal       0.56      0.35      0.43       548

    accuracy                           0.63      1924
   macro avg       0.62      0.61      0.60      1924
weighted avg       0.62      0.63      0.62      1924



In [14]:
# 4. Random Forest — ensemble of decision trees with bagging
# max_depth=10 limits individual tree depth to reduce overfitting
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_acc, rf_pred = train_and_evaluate(rf_model, X_train, y_train, X_test, y_test, "Random Forest")


=== Random Forest ===
Accuracy: 0.5598
              precision    recall  f1-score   support

        hate       0.56      0.65      0.60       594
   offensive       0.56      0.83      0.67       782
      normal       0.55      0.07      0.13       548

    accuracy                           0.56      1924
   macro avg       0.56      0.52      0.47      1924
weighted avg       0.56      0.56      0.50      1924



In [ ]:
# Summary of results
print("\n" + "="*50)
print("SUMMARY - Traditional ML on RoBERTa Embeddings")
print("="*50)
print(f"XGBoost:          {xgb_acc:.4f}")
print(f"MLP (sklearn):    {mlp_acc:.4f}")
print(f"MLP (PyTorch):    {pytorch_mlp_acc:.4f}")
print(f"Logistic Reg:     {lr_acc:.4f}")
print(f"Random Forest:    {rf_acc:.4f}")


SUMMARY - Traditional ML on RoBERTa Embeddings
XGBoost:          0.6034
MLP:              0.6154
Logistic Reg:     0.6299
Random Forest:    0.5598


In [22]:
# Persist embeddings to disk so downstream notebooks (e.g. analysis, binary)
# can load them without re-running RoBERTa inference
os.makedirs('embeddings', exist_ok=True)

np.save('embeddings/X_train_roberta.npy', X_train)
np.save('embeddings/y_train.npy', y_train)
np.save('embeddings/X_valid_roberta.npy', X_valid)
np.save('embeddings/y_valid.npy', y_valid)
np.save('embeddings/X_test_roberta.npy', X_test)
np.save('embeddings/y_test.npy', y_test)

print("Embeddings saved to embeddings/ folder")
print(f"Embedding dimension: {X_train.shape[1]}")

Embeddings saved to embeddings/ folder
Embedding dimension: 768
